 # ITEC6327 - Trí tuệ nhân tạo nâng cao (TH1004-AI2301C)
 ## Lab 4: N-Gram Language Models
 Creator: Phạm Hoàng An ✨

 Hope you have a good time 😎

## 1. Introduction to N-Grams and Counting

### 1.1 Defining N-Grams
**Theory:**
A **language model (LM)** is a model that assigns probabilities to sequences of words. The simplest version of this is the **n-gram**, which is a sequence of $n$ words.
*   A **bigram** (2-gram) is a two-word sequence like "please turn".
*   A **trigram** (3-gram) is a three-word sequence like "please turn your".

Using Python's `zip` function is an efficient way to create n-grams from a list of tokens.

In [ ]:
tokens = ["i", "am", "sam"]
bigrams = list(zip(tokens, tokens[1:]))
print(bigrams) # Output: [('i', 'am'), ('am', 'sam')]

### 1.2.Exercise 1: Implementing N-Gram
Implement a generic function that takes a list of tokens and an integer `n` to return a list of all n-gram tuples.

In [ ]:
def get_ngrams(tokens, n):
    """
    Input: list of strings (tokens), integer n
    Output: list of tuples representing n-grams
    """
    # YOUR CODE HERE

### 1.3 Sentence Augmentation (Markers)
**Theory:**
To properly estimate the probability of the first word in a sentence, we need to provide it with a "history" or context. We do this by augmenting each sentence with a special start symbol **`<s>`**. Similarly, we add an end-symbol **`</s>`** to allow the model to learn when a sequence is likely to end. For example, a trigram model at the start of a sentence would use two start symbols: `P(I | <s> <s>)`.

In [ ]:
sentence = "i am sam"
augmented = f"<s> {sentence} </s>"
tokens = augmented.split()
print(tokens) # Output: ['<s>', 'i', 'am', 'sam', '</s>']

### 1.4. Exercise 2:
Write a function that takes a raw string and returns a list of tokens augmented with start and end symbols.

In [ ]:
def preprocess_sentence(text, n):
    """
    Input: raw string, n (for n-gram context)
    Output: list of tokens with n-1 <s> markers and one </s> marker
    """
    # YOUR CODE HERE

### 1.5 The Chain Rule and Markov Assumption

**Theory:**
The **chain rule of probability** allows us to compute the joint probability of an entire sequence $P(w_1 \dots w_n)$ by multiplying the conditional probabilities of each word given all preceding words. However, calculating this for long sequences is often impossible because many contexts may never have occurred before.

To solve this, we use the **Markov assumption**: the probability of a word depends only on the previous $n-1$ words. For a **bigram model**, we approximate the probability of a word given its entire history as just the probability given the single previous word:
$P(w_n | w_{1:n-1}) \approx P(w_n | w_{n-1})$.

**Python Sample:**
We often use nested dictionaries to store counts of n-grams, where the first key is the history and the second key is the target word.

In [ ]:
counts = {"<s>": {"i": 2, "sam": 1}}
print(counts["<s>"]["i"]) # Returns the count of bigram (<s>, i)

### 1.6. Exercise 3:
Create a function that takes a list of augmented tokens and populates a dictionary with bigram counts.



In [ ]:
def count_bigrams(augmented_tokens):
    """
    Input: list of tokens (including markers)
    Output: nested dictionary {history: {word: count}}
    """
    counts = {}
    # YOUR CODE HERE

    return counts

## 2. Maximum Likelihood Estimation (MLE) and Log Probabilities


### 2.1 Theory: Estimating Probabilities via Relative Frequency

To estimate the probability of a word given its history, we use **Maximum Likelihood Estimation (MLE)**. This involves taking a large corpus, counting the number of times we see a specific history (like the bigram $w_{n-1}$), and seeing how often it is followed by the word $w_n$. We **normalize** these counts so that the resulting probabilities fall between 0 and 1. The formula for bigram probabilities is:
$$P(w_n | w_{n-1}) = \frac{C(w_{n-1}w_n)}{C(w_{n-1})}$$
Where $C(w_{n-1}w_n)$ is the count of the bigram and $C(w_{n-1})$ is the count of the unigram (the prefix).

In [ ]:
# Calculating P(Sam | am) from a small count dictionary
bigram_counts = {("i", "am"): 2, ("am", "sam"): 1, ("am", "happy"): 1}
unigram_counts = {"am": 2}

# MLE Calculation
prob = bigram_counts[("am", "sam")] / unigram_counts["am"]
print(f"P(sam|am) = {prob}") # Output: 0.5

### 2.2 Exercise: Bigram Probability Calculator
Using the counts generated in Part 1, implement a function that calculates the MLE probability for a specific word given a single word of history. If the history has never been seen (count is 0), return 0.0 for now.

In [ ]:
def calculate_bigram_prob(word, history, bigram_counts, unigram_counts):
    """
    Input: target word, history word, bigram count dict, unigram count dict
    Output: Float probability
    """
    # YOUR CODE HERE

    return 0.0

### 2.3. Theory: Calculating the Probability of an Entire Sentence
The **chain rule of probability** allows us to compute the joint probability of a sequence by multiplying the conditional probabilities of each word. Under the **Markov assumption** for a bigram model, we approximate the probability of a sentence $W$ as the product of the probability of each word given its predecessor:
$$P(w_{1:n}) \approx \prod_{k=1}^{n} P(w_k | w_{k-1})$$
This allows us to estimate how likely a specific sentence (like "I want English food") is compared to another.

In [ ]:
# Multiplying raw probabilities for: <s> i want food </s>
probs = [0.25, 0.33, 0.0011, 0.5, 0.68] # Example values from source
sentence_prob = 1.0
for p in probs:
    sentence_prob *= p
print(sentence_prob) # Output: 0.000031

### 2.4 Exercise: Sentence Likelihood Implementation
Write a function that takes a list of tokens (already processed with markers) and uses your `calculate_bigram_prob` function to find the total probability of the sequence.

In [ ]:
def get_sentence_probability(tokens, bigram_counts, unigram_counts):
    """
    Input: List of tokens (e.g., ['<s>', 'i', 'am', 'sam', '</s>'])
    Output: Total probability of the sequence
    """
    # YOUR CODE HERE

    return 0.0

### 2.5 Theory: Handling Numerical Underflow with Log Probabilities
In practice, multiplying many small probabilities (all $\le 1$) leads to **numerical underflow**, where the product becomes too small for a computer to represent accurately. To solve this, we perform all computations in **log format**. Since $\log(a \times b) = \log a + \log b$, we can add log probabilities instead of multiplying raw ones. The final result can be converted back using the exponent:
$$p_1 \times p_2 \times p_3 \times p_4 = \exp(\log p_1 + \log p_2 + \log p_3 + \log p_4)$$

### 2.6 Exercise: Log-Space Probability Calculator
Re-implement your sentence probability function to work in log-space. This function should return the **sum of log probabilities**. Note: You must handle the case where a probability is 0 (as $\log(0)$ is undefined) by returning a very small number or a specific flag.

In [ ]:
import math

def get_log_probability(tokens, bigram_counts, unigram_counts):
    """
    Input: List of tokens
    Output: Sum of log probabilities
    """
    # YOUR CODE HERE

    return 0.0

## 3. Evaluation and Perplexity


### 3.1 Theory: Evaluation Metrics (Extrinsic vs. Intrinsic)
The best way to evaluate a language model's performance is to embed it in an application (like speech recognition) and see how much the application improves; this is called **extrinsic evaluation**. However, because running end-to-end systems is expensive, we often use **intrinsic evaluation**, which measures model quality independent of any application using a **test set**. A better model is one that assigns a **higher probability** to the unseen test set, meaning it more accurately predicts the data.

In a typical workflow, we split our data to ensures we aren't "training on the test set," which would create artificially high probabilities.

In [ ]:
data = ["sentence 1", "sentence 2", "sentence 3", "sentence 4", "sentence 5"]
split_idx = int(len(data) * 0.8)
train_set = data[:split_idx]
test_set = data[split_idx:]
print(f"Train size: {len(train_set)}, Test size: {len(test_set)}")

### 3.2 Exercise: Splitting the Corpus
Implement a function to shuffle a list of sentences and split them into a training and testing corpus based on a provided ratio.

In [ ]:
import random

def split_corpus(sentences, train_ratio=0.8):
    """
    Input: List of sentences, float ratio for training
    Output: Tuple of (train_list, test_list)
    """
    # YOUR CODE HERE

    return train_list, test_list

### 3.3 Theory: Perplexity ($PPL$)
While we want high probabilities for test sets, we usually use a variant called **perplexity** for evaluation. The perplexity of a language model on a test set is the **inverse probability** of that test set, normalized by the number of words $N$.
The formula for a sequence $W$ is:
$$PPL(W) = \sqrt[N]{\frac{1}{P(w_1w_2\dots w_N)}}$$
Under the bigram assumption, this is the geometric mean of the inverse bigram probabilities. A **lower perplexity** indicates a better model that is "less surprised" by the test data.

**Python Sample:**
If you have a vocabulary of 10 digits that each occur with equal probability ($P=1/10$), the perplexity is exactly 10.

In [ ]:
import math
# Example: 10 words, each with prob 0.1, N=10
prob_each = 0.1
N = 10
# Joint probability P(W) = 0.1^10
joint_prob = math.pow(prob_each, N)
# Perplexity = (1 / joint_prob) ^ (1/N)
perplexity = math.pow(1.0 / joint_prob, 1.0/N)
print(f"Perplexity: {perplexity}") # Output: 10.0

### 3.4 Exercise: Implementing Perplexity
Write a function to calculate the perplexity of a test sentence using your bigram model. You should use your previously created `get_log_probability` function to avoid numerical underflow.
*Note: Perplexity is calculated as $\exp(-\frac{1}{N} \times \text{sum\_of\_log\_probs})$.*

In [ ]:
def calculate_perplexity(test_tokens, bigram_counts, unigram_counts):
    """
    Input: List of tokens (test set), count dictionaries
    Output: Float perplexity score
    """
    N = len(test_tokens) # Number of words including </s> but not <s>
    # YOUR CODE HERE

    return 0.0

### 3.5 Theory: The Branching Factor
Perplexity can be thought of as the **weighted average branching factor** of a language. The branching factor is the number of possible next words that can follow any given word. For example, if a model has a perplexity of 10, it means that at each point where it has to predict a word, it is as if the model had to choose among 10 equivalent possibilities.

This demonstrates how a "predictable" sequence (high probability) leads to lower perplexity.

In [ ]:
# Model A is uncertain (P=0.1 for 10 words) -> PPL = 10
# Model B is certain (P=0.8 for one word, small probs for others) -> PPL will be much lower
log_probs_certain = [math.log(0.8), math.log(0.02), math.log(0.02)]
avg_log_prob = sum(log_probs_certain) / len(log_probs_certain)
ppl_certain = math.exp(-avg_log_prob)
print(f"Predictable Model PPL: {ppl_certain:.2f}")

### 3.6 Exercise: Comparative Analysis
Write a short script that calculates the perplexity for two different sentences: one that is common (e.g., "I am Sam") and one that is rare or syntactically strange (e.g., "Sam do green"). Use the counts provided in the earlier parts of the lab.


In [ ]:
# Use your calculate_perplexity function on two different token lists
# Compare the results and print which one has a lower (better) score.

# YOUR CODE HERE

## 4. Smoothing Techniques


### 4.1 Theory: The Zero Probability Problem
A major challenge with N-gram models is **sparsity**. Because language is creative, we often encounter N-grams in a test set that never appeared in our training data. If a single bigram in a test sentence has a count of zero, the **Maximum Likelihood Estimate (MLE)** will assign it a probability of 0.0. Since we multiply these probabilities together to get the sentence likelihood (or add them in log-space), one zero makes the entire sequence probability zero, resulting in an **infinite perplexity**. **Smoothing** (or **discounting**) solves this by shaving off a bit of probability mass from frequent events and assigning it to unseen events.

This demonstrates how a single unseen bigram can "break" the probability calculation of a sentence.

In [ ]:
# Probs for "i want chinese food" where 'chinese' was never seen after 'want'
probs = [0.25, 0.33, 0.0, 0.5, 0.68]
sentence_prob = 1.0
for p in probs:
    sentence_prob *= p
print(f"Total Probability: {sentence_prob}") # Output: 0.0

### 4.2 Exercise: Identifying Unseen Bigrams
Write a function that compares a test set against a training count dictionary and identifies which bigrams in the test set have a count of zero.

In [ ]:
def find_unseen_bigrams(test_tokens, training_bigram_counts):
    """
    Input: List of test tokens, dictionary of training counts
    Output: List of bigram tuples that were not found in training
    """
    # YOUR CODE HERE

    return unseen

### 4.3 Theory: Laplace (Add-one) Smoothing
The simplest way to do smoothing is **Laplace smoothing**, also called **add-one smoothing**. We simply add one to every possible N-gram count before normalizing. For a unigram model, the adjusted probability is:
$$P_{Laplace}(w_i) = \frac{c_i + 1}{N + V}$$
Where $c_i$ is the original count, $N$ is the total number of tokens, and **$V$ is the vocabulary size** (the number of unique word types). We must add $V$ to the denominator to ensure that the sum of all probabilities still equals 1.

Calculating the smoothed probability for a word that appeared 0 times in a vocabulary of 1000 words.

In [ ]:
count_word = 0
total_tokens_N = 10000
vocab_size_V = 1000

# Laplace smoothed probability
smoothed_p = (count_word + 1) / (total_tokens_N + vocab_size_V)
print(f"Smoothed P: {smoothed_p}") # No longer zero!

### 4.4 Exercise: Bigram Laplace Implementation
Implement a function to calculate the Laplace-smoothed bigram probability. The formula for a bigram is $P(w_n | w_{n-1}) = \frac{C(w_{n-1}w_n) + 1}{C(w_{n-1}) + V}$.

In [ ]:
def calculate_laplace_bigram_prob(word, history, bigram_counts, unigram_counts, V):
    """
    Input: target word, history word, count dicts, vocabulary size V
    Output: Smoothed float probability
    """
    # YOUR CODE HERE

    return 0.0

### 4.5 Theory: Add-k Smoothing
One drawback of Laplace smoothing is that it can move too much probability mass to unseen events. **Add-k smoothing** is an alternative where instead of adding 1, we add a fractional count **$k$** (such as 0.5, 0.05, or 0.01). This allows us to move less of the "probability mass" from seen to unseen events. The formula is:
$$P_{Add-k}(w_n | w_{n-1}) = \frac{C(w_{n-1}w_n) + k}{C(w_{n-1}) + kV}$$
The value of $k$ is typically chosen by optimizing performance on a **development set (devset)**.

Contrasting the effect of $k$ on a zero-count bigram.

In [ ]:
count_bigram = 0
count_history = 10
V = 1000
k_laplace = 1.0
k_small = 0.01

p_laplace = (count_bigram + k_laplace) / (count_history + k_laplace * V)
p_add_k = (count_bigram + k_small) / (count_history + k_small * V)

print(f"Laplace: {p_laplace:.6f} vs Add-k: {p_add_k:.6f}")

### 4.6 Exercise: Comparative Smoothing Experiment
Write a script that calculates the probability of a specific unseen bigram using three different $k$ values: $1.0$, $0.1$, and $0.01$. Print the results and observe how the probability changes as $k$ gets smaller.

In [ ]:
# Define your test case (unseen bigram)
# Define your V and count_history
# Calculate and print for k = 1.0, 0.1, 0.01

# YOUR CODE HERE

## 5. Project — The Generative Language Workbench

In this comprehensive final task, you will consolidate everything you have learned to build, evaluate, and compare three different language models: **Unigram, Bigram, and Trigram**.

**Project Requirements:**

1.  **Data Preparation & Vocabulary Management:**
    *   Load a text corpus and split it into **Training (80%)** and **Test (20%)** sets.
    *   Handle **Unknown Words (`<UNK>`)**: Identify words in the training set that appear only once (or below a threshold) and replace them with `<UNK>`. Any word in the test set not seen in training must also be treated as `<UNK>`.
    *   **Augmentation**: For your trigram model, ensure you add two start markers `<s> <s>` and one end marker `</s>` to every sentence.

2.  **Model Implementation:**
    *   Create a system that calculates counts for all three N-gram levels.
    *   Implement **Add-k Smoothing** across all models. You should experiment with at least two different values of $k$ (e.g., $k=1.0$ and $k=0.01$) to see how it affects the results.

3.  **Evaluation and Comparison:**
    *   Calculate the **Perplexity** for the Unigram, Bigram, and Trigram models on the test set.
    *   Your results should demonstrate that the Trigram model generally achieves a lower (better) perplexity because it is "less surprised" by the test sequence.

4.  **Generative Sampling:**
    *   Implement a **sampling** engine. For each model, generate 5 random sentences.
    *   Sampling works by choosing a random next word according to the probability distribution defined by your model's current context. Start with the appropriate number of `<s>` markers and stop when `</s>` is generated.

5.  **Final Analysis:**
    *   Provide a written summary comparing the generated output. Which model sounds most like natural English? Why does the Trigram model struggle more with "zero counts" than the Unigram model before smoothing is applied?

In [ ]:
# --- PROJECT IMPLEMENTATION SPACE ---
# Use this space to build your NGramModel class,
# train on a real corpus, and run your evaluation/generation loop.

class LanguageModelWorkbench:
    def __init__(self, n, k):
        self.n = n
        self.k = k
        # Initialization code

    def train(self, corpus):
        # Preprocessing, <UNK> handling, and counting
        pass

    def get_perplexity(self, test_data):
        # Log-space perplexity calculation
        pass

    def sample_sentence(self):
        # Generative sampling logic
        pass

# Run your experiment here: